# 实验三：空气数据的白化 Koopman 全链条分析

本 notebook 面向空气质量数据，延续 `exp_kuramoto_0321.ipynb` 与 `exp_map_0321.ipynb` 的整体分析框架。

与前两个实验相比，这里的两个关键变化是：

- 数据来源不再是人工模型轨道，而是空气质量实测数据；
- 观测函数不采用 Fourier 或 polynomial，而采用时间延迟嵌入。

整体逻辑保持不变：

原始空气数据 -> 空间子集选择 -> 状态矩阵构造 -> 时间延迟 lift -> 协方差矩阵 -> $K_{\mathrm{raw}}$ 与 $\bar K$ -> 奇异值谱与 EC -> 粗粒化函数与宏观变量。

## Block 1. 环境导入与统一风格

这一部分完成以下准备工作：

- 导入通用库与绘图工具；
- 将仓库根目录加入 `sys.path`；
- 从 `tools.py` 中导入 Koopman 主链函数与空气数据辅助函数；
- 统一 notebook 的绘图风格；
- 定义少量本地辅助函数，便于热图和标准化展示。

这一块对应前两个实验中的公共配置区。

In [ ]:

import sys
from pathlib import Path

REPO_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / 'tools.py').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repository root containing tools.py')
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import numpy as np
if not hasattr(np, 'typeDict'):
    np.typeDict = np.sctypeDict
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import pysindy as ps

from tools import (
    compute_transition_covariances,
    fit_data_koopman_operator,
    get_positive_contributions,
    compute_entropy,
    init_artifacts,
    analyze_kbar_metrics,
    build_macro_from_kbar,
    print_summary,
    compute_gamma_ce_metrics,
    lift_time_delay,
    load_air_data_subset,
    build_air_feature_matrix,
    summarize_air_subset,
    split_and_group_matrices,
    plot_station_with_map,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei',
    'SimHei',
    'Noto Sans CJK SC',
    'Source Han Sans SC',
    'Arial Unicode MS',
    'DejaVu Sans',
]

HEATMAP_CMAP = 'vlag'


def sparse_labels(labels, step=1):
    if labels is None:
        return False
    if step <= 1:
        return labels
    return [label if idx % step == 0 else '' for idx, label in enumerate(labels)]


def plot_matrix_heatmap(matrix, title, row_labels=None, col_labels=None, center=0.0, figsize=(6, 6), label_step=1, cmap=HEATMAP_CMAP):
    matrix = np.asarray(matrix)
    n_rows, n_cols = matrix.shape
    is_square = n_rows == n_cols

    if is_square:
        plot_size = (6, 6)
        square = True
        x_step = max(label_step, max(1, int(np.ceil(n_cols / 10))))
        y_step = max(label_step, max(1, int(np.ceil(n_rows / 10))))
    else:
        plot_size = figsize
        square = False
        x_step = max(label_step, max(1, int(np.ceil(n_cols / 8))))
        y_step = max(label_step, max(1, int(np.ceil(n_rows / 18))))

    fig, ax = plt.subplots(figsize=plot_size)
    sns.heatmap(
        matrix,
        ax=ax,
        cmap=cmap,
        center=center,
        xticklabels=sparse_labels(col_labels, x_step),
        yticklabels=sparse_labels(row_labels, y_step),
        square=square,
        cbar_kws={'shrink': 0.72, 'fraction': 0.035, 'pad': 0.02},
    )
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90, labelsize=6.5)
    ax.tick_params(axis='y', rotation=0, labelsize=6.5)
    plt.tight_layout()
    plt.show()


def standardize_for_plot(x):
    x = np.asarray(x, dtype=float)
    return (x - np.mean(x)) / (np.std(x) + 1e-12)


def locate_existing_path(candidates, label):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    raise FileNotFoundError(f'Could not locate {label}. Tried: {candidates}')


## Block 2. 全局参数配置

这一部分统一管理实验参数，避免后续代码中出现大量硬编码。

参数主要分为几类：

- 数据路径与站点元数据路径；
- 空间筛选参数，如全部站点、某些省、某些市；
- 变量选择，如 `pm25`、`o3`；
- 时间延迟参数 `n_delays` 与 `delay_interval`；
- 白化 Koopman 主链参数，如 `lag_steps`、`alpha`、`eps`、`ridge`；
- 可视化参数，如代表性站点数量、热图标签间隔、地图布局列数。

其中：

- `delay_interval` 控制延迟层之间的时间间隔；
- `lag_steps` 控制样本对 $\left(x_t, x_{t+\tau}\right)$ 中向前推进的步长；
- 当 `n_delays > 0` 时，lift 后维度会扩大到原来的 $n_{\mathrm{delays}} + 1$ 倍。

In [ ]:

config = {
    'experiment_name': 'exp_air_0321',
    'dataset_path_candidates': [
        REPO_ROOT / 'data' / 'dataset_yrd.nc',
        REPO_ROOT / 'data' / 'dataset_yrd.nc',
    ],
    'station_meta_path_candidates': [
        REPO_ROOT / 'data' / 'stations_yrd.csv',
        REPO_ROOT / 'data' / 'stations_yrd.csv',
    ],
    'subset': {
        'subset_mode': 'all',
        'province_names': None,
        'city_names': None,
        'station_ids': None,
        'time_slice': None,
        # 示例 1：只用一个省
        # 'subset_mode': 'province',
        # 'province_names': ['江苏省'],
        # 示例 2：只用几个市
        # 'subset_mode': 'cities',
        # 'city_names': ['上海市', '南京市', '苏州市'],
    },
    'variables': ['PM2.5', 'O3'],
    'observable_mode': 'time-delay',
    'n_delays': 2,
    'delay_interval': 24,
    'burn_in_steps': 0,
    'sample_stride': 1,
    'lag_steps': 24,
    'rank': 1,
    'macro_r_mode': 'selected',
    'alpha': 1.0,
    'eps': 1e-10,
    'ridge': 1e-8,
    'rank_candidates': None,
    'selected_station_plot_count': 3,
    'heatmap_label_step': 1,
    'map_ncols': 2,
}

artifacts = init_artifacts(config)
config


## Block 3. 空气数据读取与空间子集选择

这一部分调用 `load_air_data_subset(...)` 与 `summarize_air_subset(...)`：

- 读取空气质量数据；
- 读取站点元数据；
- 按配置筛选空间子集；
- 汇总所选省、市、站点数量与时间长度。

这样做的目的是把数据入口统一下来，使得后面切换研究区域时，不需要改动 Koopman 主链。

常见配置示例：

- 使用全部站点：`subset_mode='all'`；
- 只使用一个省：`subset_mode='province'`，例如 `province_names=['江苏省']`；
- 只使用几个市：`subset_mode='cities'`，例如 `city_names=['上海市', '南京市', '苏州市']`。

In [ ]:

dataset_path = locate_existing_path(config['dataset_path_candidates'], 'air dataset')
station_meta_path = locate_existing_path(config['station_meta_path_candidates'], 'station metadata')

air_subset = load_air_data_subset(
    dataset_path=dataset_path,
    station_meta_path=station_meta_path,
    subset_mode=config['subset']['subset_mode'],
    province_names=config['subset']['province_names'],
    city_names=config['subset']['city_names'],
    station_ids=config['subset']['station_ids'],
    variables=config['variables'],
    time_slice=config['subset']['time_slice'],
)

subset_summary = summarize_air_subset(air_subset)
station_meta_subset = air_subset['station_meta_subset']

display(subset_summary['summary_df'])
display(subset_summary['province_counts'])
display(subset_summary['city_counts'].head(20))

print('Dataset path:', dataset_path)
print('Station meta path:', station_meta_path)
print('Selected station count:', len(station_meta_subset))
print('Selected variables:', air_subset['selected_variables'])


## Block 4. 原始状态矩阵构造与基础展示

这一部分把所选空气数据整理成统一的微观状态矩阵：

$$
X_{\mathrm{raw}} \in \mathbb{R}^{T \times d}.
$$

`build_air_feature_matrix(...)` 会返回：

- `x_data_raw`；
- `feature_names_raw`；
- `times_raw`；
- `station_meta_raw`；
- 各个变量对应的列切片信息。

同时，这里会展示少量代表性站点的微观时间序列，避免图像过于拥挤。

In [ ]:

air_matrix = build_air_feature_matrix(air_subset)

x_data_raw = air_matrix['x_data_raw']
t_data_raw = air_matrix['times']
station_meta_raw = air_matrix['station_meta']
station_ids_raw = air_matrix['station_ids']
variable_slices_raw = air_matrix['variable_slices']

variable_name_alias = {
    'PM2.5': 'pm25',
    'O3': 'o3',
}

def normalize_air_variable_name(name):
    name = str(name)
    if name in variable_name_alias:
        return variable_name_alias[name]
    return name.lower().replace('.', '').replace(' ', '_')

feature_names_raw = []
for feature_name in air_matrix['feature_names']:
    normalized_name = str(feature_name)
    for raw_var, alias_var in variable_name_alias.items():
        normalized_name = normalized_name.replace(f'_{raw_var}', f'_{alias_var}')
    feature_names_raw.append(normalized_name)

variables_normalized = [normalize_air_variable_name(var) for var in air_matrix['variables']]
variable_slices = {
    normalize_air_variable_name(var): slc
    for var, slc in variable_slices_raw.items()
}

artifacts['raw'] = {
    'air_subset': air_subset,
    'air_matrix': air_matrix,
    'x_data_raw': x_data_raw,
    'feature_names_raw': feature_names_raw,
    't_data_raw': t_data_raw,
    'variables_dataset': air_matrix['variables'],
    'variables_normalized': variables_normalized,
}

print('Raw data shape:', x_data_raw.shape)
print('Feature dimension:', x_data_raw.shape[1])
print('Number of stations:', air_matrix['n_stations'])
print('Dataset variables:', air_matrix['variables'])
print('Normalized variables:', variables_normalized)
print('First 10 normalized feature names:', feature_names_raw[:10])

display(station_meta_raw.head())

selected_station_count = min(config['selected_station_plot_count'], len(station_ids_raw))
selected_station_ids = station_ids_raw[:selected_station_count]
selected_micro_indices = []
for raw_var in air_matrix['variables']:
    var_start = variable_slices_raw[raw_var].start
    selected_micro_indices.extend([var_start + i for i in range(selected_station_count)])
selected_micro_indices = selected_micro_indices[: min(6, len(selected_micro_indices))]

plt.figure(figsize=(12, 4.5))
for idx in selected_micro_indices:
    plt.plot(t_data_raw, x_data_raw[:, idx], linewidth=1.0, label=feature_names_raw[idx])
plt.title('Micro trajectories before delay embedding')
plt.xlabel('Time')
plt.ylabel('Concentration')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()


## Block 5. 观测函数：时间延迟嵌入

空气实验中的观测函数不再使用 Fourier 或 polynomial，而是采用时间延迟嵌入：

$$
\chi(x_t) = [x_t, x_{t-\Delta}, x_{t-2\Delta}, \dots, x_{t-m\Delta}].
$$

这样做的原因是空气质量过程具有明显的时间记忆性，用延迟嵌入把历史信息显式纳入扩展状态更自然。

这一部分调用 `lift_time_delay(...)`，并保留延迟后的特征名，方便后面解释左奇异向量和地图投影。

In [ ]:

burn_in_steps = config['burn_in_steps']
sample_stride = config['sample_stride']

x_data_fit = x_data_raw[burn_in_steps::sample_stride].copy()
t_data_fit = t_data_raw[burn_in_steps::sample_stride].copy()
feature_names_fit = feature_names_raw.copy()
dt_fit = sample_stride

H_list, delay_feature_names = lift_time_delay(
    x_data_fit,
    feature_names=feature_names_fit,
    n_delays=config['n_delays'],
    delay_interval=config['delay_interval'],
)

if isinstance(H_list, list):
    if len(H_list) != 1:
        raise ValueError(f'Expected a single delayed trajectory, got {len(H_list)} trajectories')
    x_data_lift = np.asarray(H_list[0], dtype=float)
else:
    x_data_lift = np.asarray(H_list, dtype=float)

x_data_lift_centered = np.asarray(x_data_lift - np.mean(x_data_lift, axis=0, keepdims=True), dtype=float)
x_data_lift = np.asarray(x_data_lift_centered, dtype=float)
raw_names = delay_feature_names

t_data_lift = t_data_fit[config['n_delays'] * config['delay_interval']:]
tau = config['lag_steps'] * dt_fit

artifacts['prep'] = {
    'x_data_fit': x_data_fit,
    't_data_fit': t_data_fit,
    'feature_names_fit': feature_names_fit,
    'dt_fit': dt_fit,
    'tau': tau,
}
artifacts['obs'] = {
    'x_data_lift': x_data_lift,
    'x_data_lift_centered': x_data_lift_centered,
    'feature_names': raw_names,
    'observable_mode': config['observable_mode'],
}

print('Observable mode:', config['observable_mode'])
print('Prepared raw shape:', x_data_fit.shape)
print('Lifted shape:', x_data_lift.shape)
print('n_delays:', config['n_delays'])
print('delay_interval:', config['delay_interval'])
print('lag_steps:', config['lag_steps'])
print('tau =', tau)
print('First 12 lifted feature names:', raw_names[:12])


## Block 6. 计算 $C_{00}$、$C_{01}$ 与 $C_{11}$

这一部分进入白化 Koopman 链条的核心统计层。虽然观测函数改成了时间延迟嵌入，但主链本身没有变化，仍然构造：

$$
C_{00} = \mathbb{E}[\tilde X_t \tilde X_t^{\top}],
\qquad
C_{01} = \mathbb{E}[\tilde X_t \tilde Y_t^{\top}],
\qquad
C_{11} = \mathbb{E}[\tilde Y_t \tilde Y_t^{\top}].
$$

这里的 $\tilde X_t$ 与 $\tilde Y_t$ 都处在延迟扩展状态空间中，因此这些矩阵同时编码了：

- 空间站点之间的相关性；
- 不同污染物变量之间的相关性；
- 不同延迟层之间的相关性。

In [ ]:

transition_stats = compute_transition_covariances(
    [x_data_lift],
    library=None,
    weights='uniform',
    lag_steps=config['lag_steps'],
)

C00 = transition_stats['C00']
C01 = transition_stats['C01']
C11 = transition_stats['C11']
X_pairs = transition_stats['X']
Y_pairs = transition_stats['Y']

artifacts['cov'] = {
    'transition_stats': transition_stats,
    'C00': C00,
    'C01': C01,
    'C11': C11,
    'X_pairs': X_pairs,
    'Y_pairs': Y_pairs,
}

print(f"lag_steps = {config['lag_steps']}, tau = {tau:.6f}")
print('Pair count:', X_pairs.shape[0])
print('C00 shape:', C00.shape)
print('C01 shape:', C01.shape)
print('C11 shape:', C11.shape)
print('cond(C00):', np.linalg.cond(C00))
print('cond(C11):', np.linalg.cond(C11))
print('min eig(C00):', np.min(np.linalg.eigvalsh(0.5 * (C00 + C00.T))))
print('min eig(C11):', np.min(np.linalg.eigvalsh(0.5 * (C11 + C11.T))))

plot_matrix_heatmap(C00, 'C00', row_labels=raw_names, col_labels=raw_names, figsize=(6, 6), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(C01, 'C01', row_labels=raw_names, col_labels=raw_names, figsize=(6, 6), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(C11, 'C11', row_labels=raw_names, col_labels=raw_names, figsize=(6, 6), label_step=config['heatmap_label_step'])


## Block 7. 构造 $K_{\mathrm{raw}}$ 与 $\bar K$

由协方差矩阵构造：

$$
K_{\mathrm{raw}} = C_{00}^{\dagger} C_{01},
\qquad
\bar K = C_{00}^{-1/2} C_{01} C_{11}^{-1/2}.
$$

在本实验中：

- `K_raw` 仅用于奇异值谱对照；
- `K_bar` 是后续分析的核心对象；
- 对空气实验来说，$\bar K$ 定义在时间延迟 lift 后的扩展状态空间上。

In [ ]:

koop_fit = fit_data_koopman_operator(
    [x_data_lift],
    library=None,
    weights='uniform',
    eps=config['eps'],
    ridge=config['ridge'],
    lag_steps=config['lag_steps'],
)

K_raw = koop_fit['A']
K_bar = koop_fit['K_bar']
K_bar_from_A = koop_fit['K_bar_from_A']
C00_inv_sqrt = koop_fit['C00_inv_sqrt']
C11_inv_sqrt = koop_fit['C11_inv_sqrt']

artifacts['koopman'] = {
    'koop_fit': koop_fit,
    'K_raw': K_raw,
    'K_bar': K_bar,
    'K_bar_from_A': K_bar_from_A,
    'C00_inv_sqrt': C00_inv_sqrt,
    'C11_inv_sqrt': C11_inv_sqrt,
}

print('K_raw shape:', K_raw.shape)
print('K_bar shape:', K_bar.shape)
print('||K_bar - K_bar_from_A||_F =', np.linalg.norm(K_bar - K_bar_from_A))
print('sigma_max(K_bar) <= 1, approx =', np.linalg.svd(K_bar, compute_uv=False)[0])

plot_matrix_heatmap(K_raw, 'K_raw', row_labels=raw_names, col_labels=raw_names, figsize=(6, 6), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(K_bar, 'K_bar', row_labels=raw_names, col_labels=raw_names, figsize=(6, 6), label_step=config['heatmap_label_step'])



## Block 8. 奇异值谱与 EC

这一部分对 $\bar K$ 做奇异值分解：

$$
\bar K = U \Sigma V^{\top}.
$$

同时也会计算 `K_raw` 的奇异值谱，作为对照。这里共绘制四张谱图：

1. `K_bar` 的奇异值谱；
2. `K_raw` 的奇异值谱；
3. `K_bar` 与 `K_raw` 的整体对比图；
4. 前 10% 奇异值的双坐标对比图。

随后基于 $\bar K$ 的奇异值谱继续计算 EC 增量序列和 EC 标量值。


In [ ]:

U_raw, S_raw, Vt_raw = np.linalg.svd(K_raw, full_matrices=False)
metrics = analyze_kbar_metrics(K_bar, alpha=config['alpha'], eps=config['eps'])

U_bar = metrics['U']
S_bar = metrics['S']
Vt_bar = metrics['Vt']

diff = get_positive_contributions(S_bar)
EC = compute_entropy(diff)

top10_count = max(1, int(np.ceil(0.1 * len(S_bar))))

artifacts['spectral'] = {
    'U_raw': U_raw,
    'S_raw': S_raw,
    'Vt_raw': Vt_raw,
    'U_bar': U_bar,
    'S_bar': S_bar,
    'Vt_bar': Vt_bar,
    'diff': diff,
    'EC': EC,
    'top10_count': top10_count,
}
artifacts['metrics']['kbar'] = metrics
artifacts['metrics']['EC'] = EC
artifacts['metrics']['EC_diff'] = diff

print('Top 10 singular values of K_bar:', S_bar[:10])
print('Top 10 singular values of K_raw:', S_raw[:10])
print('EC entropy:', EC)
print('Top 10% singular-value count:', top10_count)

idx_bar = np.arange(1, len(S_bar) + 1)
idx_raw = np.arange(1, len(S_raw) + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_bar, S_bar, marker='o', linewidth=1.8, label='K_bar singular values', color='tab:blue')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum: K_bar')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_raw, S_raw, marker='s', linewidth=1.4, linestyle='--', label='K_raw singular values', color='0.35')
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum: K_raw')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idx_bar, S_bar, marker='o', linewidth=1.8, label='K_bar singular values', color='tab:blue')
ax.plot(idx_raw, S_raw, marker='s', linewidth=1.2, linestyle='--', label='K_raw singular values', color='0.35')
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1.0)
ax.set_xlabel('Singular index')
ax.set_ylabel('Singular value')
ax.set_title('Singular spectrum comparison: K_bar vs K_raw')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax_left = plt.subplots(figsize=(8, 4.5))
ax_right = ax_left.twinx()
line1 = ax_left.plot(idx_bar[:top10_count], S_bar[:top10_count], marker='o', linewidth=1.8, label=f'K_bar top 10% ({top10_count} values)', color='tab:purple')
line2 = ax_right.plot(idx_raw[:top10_count], S_raw[:top10_count], marker='s', linewidth=1.2, linestyle='--', label=f'K_raw top 10% ({top10_count} values)', color='0.35')
ax_left.axhline(1.0, color='tab:purple', linestyle=':', linewidth=1.0, alpha=0.6)
ax_left.set_xlabel('Singular index')
ax_left.set_ylabel('K_bar singular value', color='tab:purple')
ax_right.set_ylabel('K_raw singular value', color='0.35')
ax_left.tick_params(axis='y', colors='tab:purple')
ax_right.tick_params(axis='y', colors='0.35')
ax_left.set_ylim(0.0, 1.0)
ax_left.set_title('Top 10% singular spectrum comparison')
lines = line1 + line2
labels = [line.get_label() for line in lines]
ax_left.legend(lines, labels, loc='best')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, len(diff) + 1), diff, marker='o', linewidth=1.8, color='tab:green', label=f'EC spectrum (EC={EC:.4f})')
ax.set_xlabel('Index')
ax.set_ylabel('EC increment')
ax.set_title('EC spectrum derived from K_bar singular values')
ax.legend()
plt.tight_layout()
plt.show()


## Block 9. 计算 $D_K$、$N_K$ 与主总分 $\mathcal{G}_{\alpha}^{K}$

定义：

$$
D_K = \left(I - \bar K^{\top} \bar K\right)^{-1},
\qquad
N_K = \bar K \left(I - \bar K^{\top} \bar K\right)^{-1} \bar K^{\top}.
$$

然后计算主总分：

$$
\mathcal{G}_{\alpha}^{K} = \left(\frac{1}{2} - \frac{\alpha}{4}\right) \log \operatorname{pdet}(N_K)
+ \frac{\alpha}{4} \log \operatorname{pdet}(D_K).
$$

对于空气实验，这些量衡量的是延迟扩展状态上的主导预测结构。

In [ ]:

D_K = metrics['D_K']
N_K = metrics['N_K']
G_alpha_K = metrics['G_alpha_K']

print('log pdet(D_K):', metrics['log_pdet_D_K'])
print('log pdet(N_K):', metrics['log_pdet_N_K'])
print('G_alpha^K:', G_alpha_K)

plot_matrix_heatmap(D_K, 'D_K ', center=None, figsize=(6, 6), cmap=HEATMAP_CMAP)
plot_matrix_heatmap(N_K, 'N_K ', center=0.0, figsize=(6, 6), cmap=HEATMAP_CMAP)



## Block 10. 定义 12/13 下的可逆性、维度平均可逆性与 CE

对白化 Koopman 矩阵的奇异值 $\sigma_i$，按定义 12 计算：

$$
\Gamma_\alpha^K(\bar K)=\sum_{i=1}^{d_{\mathrm{eff}}}\sigma_i^\alpha,
\qquad
\gamma_\alpha^K(\bar K)=\frac{1}{d_{\mathrm{eff}}}\Gamma_\alpha^K(\bar K).
$$

再按定义 13，对每个候选截断维数 $r$ 计算：

$$
\Delta \Gamma_\alpha^K(r)
= \frac{1}{r} \sum_{i=1}^{r} \sigma_i^\alpha
- \frac{1}{d_{\mathrm{eff}}} \sum_{i=1}^{d_{\mathrm{eff}}} \sigma_i^\alpha.
$$

这里同时输出手动指定 `rank` 对应的 CE，以及自动选择 `selected_r` 对应的 CE。


In [ ]:

manual_r = config.get('rank')
rank_candidates = config.get('rank_candidates')
gamma_metrics = compute_gamma_ce_metrics(
    S_bar,
    alpha=config['alpha'],
    rank_candidates=rank_candidates,
    manual_r=manual_r,
    eps=config['eps'],
)

reversibility_channel_scores = gamma_metrics['reversibility_channel_scores']
Gamma_alpha_K = gamma_metrics['Gamma_alpha_K']
gamma_alpha_K = gamma_metrics['gamma_alpha_K']
effective_rank_gamma = gamma_metrics['effective_rank']
rank_candidates = gamma_metrics['rank_candidates']
delta_gamma_by_r = gamma_metrics['delta_gamma_by_r']
selected_r = gamma_metrics['selected_r']
delta_gamma_selected_r = gamma_metrics['delta_gamma_selected_r']
manual_r = gamma_metrics['manual_r']
delta_gamma_manual_r = gamma_metrics['delta_gamma_manual_r']

artifacts['metrics']['reversibility_channel_scores'] = reversibility_channel_scores
artifacts['metrics']['Gamma_alpha_K'] = Gamma_alpha_K
artifacts['metrics']['gamma_alpha_K'] = gamma_alpha_K
artifacts['metrics']['delta_gamma_by_r'] = delta_gamma_by_r
artifacts['metrics']['selected_r'] = selected_r
artifacts['metrics']['delta_gamma_selected_r'] = delta_gamma_selected_r
artifacts['metrics']['delta_g_selected_r'] = delta_gamma_selected_r
artifacts['metrics']['manual_r'] = manual_r
artifacts['metrics']['delta_gamma_manual_r'] = delta_gamma_manual_r

print('Reversibility channel scores sigma_i^alpha:', reversibility_channel_scores)
print('Gamma_alpha_K:', Gamma_alpha_K)
print('gamma_alpha_K:', gamma_alpha_K)
print('effective_rank:', effective_rank_gamma)
print('Delta Gamma by r:', delta_gamma_by_r)
print('Manual r:', manual_r)
print('Delta Gamma at manual r:', delta_gamma_manual_r)
print('Selected r:', selected_r)
print('Delta Gamma at selected r (CE):', delta_gamma_selected_r)

plt.figure(figsize=(7, 4))
plt.plot(list(delta_gamma_by_r.keys()), list(delta_gamma_by_r.values()), marker='o', color='tab:blue')
if manual_r is not None:
    plt.axvline(manual_r, color='tab:orange', linestyle=':', label=f'manual r = {manual_r}')
plt.axvline(selected_r, color='tab:red', linestyle='--', label=f'selected r = {selected_r}')
plt.title('Dimension-averaged reversibility gain Delta Gamma_alpha^K(r)')
plt.xlabel('r')
plt.ylabel('Delta Gamma_alpha^K(r)')
plt.legend()
plt.tight_layout()
plt.show()



## Block 11. 左奇异向量与空间地图展示

这一部分展示：

- 完整左奇异向量矩阵 $U$；
- 用于宏观变量构造的前 $r$ 列左奇异向量；
- 绝对值结构；
- 按变量与延迟层拆分后的空间地图。

这里的 `r` 默认取自动选择的 `selected_r`，但也允许切换到手动指定的 `rank`。代码单元会打印当前采用的是哪一种。


In [ ]:

macro_r_mode = config.get('macro_r_mode', 'selected')
if macro_r_mode == 'manual' and manual_r is not None:
    macro_r = manual_r
    macro_r_source = f'manual_r = {manual_r}'
else:
    macro_r = selected_r
    macro_r_source = f'selected_r = {selected_r}'

print('Macro variables are constructed using', macro_r_source)

plot_matrix_heatmap(U_bar, 'Left singular vectors U (all)', row_labels=raw_names, col_labels=[f'u{i+1}' for i in range(U_bar.shape[1])], figsize=(7, 8), label_step=config['heatmap_label_step'])
plot_matrix_heatmap(np.abs(U_bar), 'Absolute left singular vectors |U| (all)', row_labels=raw_names, col_labels=[f'|u{i+1}|' for i in range(U_bar.shape[1])], center=None, figsize=(7, 8), label_step=config['heatmap_label_step'], cmap='Blues')
u_first_r_label_step = max(config['heatmap_label_step'], int(np.ceil(len(raw_names) / 18)))
plot_matrix_heatmap(U_bar[:, :macro_r], f'Left singular vectors U (first {macro_r})', row_labels=raw_names, col_labels=[f'u{i+1}' for i in range(macro_r)], figsize=(7, 8), label_step=u_first_r_label_step)
plot_matrix_heatmap(np.abs(U_bar[:, :macro_r]), f'Absolute left singular vectors |U| (first {macro_r})', row_labels=raw_names, col_labels=[f'|u{i+1}|' for i in range(macro_r)], center=None, figsize=(7, 8), label_step=u_first_r_label_step, cmap='Blues')

U_abs_r = np.abs(U_bar[:, :macro_r])
map_df = station_meta_raw[['station_id', 'station_name', 'city', 'province', 'lon', 'lat']].copy()

splits_pm25, splits_o3 = split_and_group_matrices(U_abs_r, raw_names, config['n_delays'] + 1)

pm25_panels = [
    {
        'data': split_df,
        'delay': delay_idx * config['delay_interval'],
        'title_suffix': '_pm25',
    }
    for delay_idx, split_df in enumerate(splits_pm25)
]
o3_panels = [
    {
        'data': split_df,
        'delay': delay_idx * config['delay_interval'],
        'title_suffix': '_o3',
    }
    for delay_idx, split_df in enumerate(splits_o3)
]

panel_values = []
for split_df in list(splits_pm25) + list(splits_o3):
    panel_values.append(np.asarray(split_df, dtype=float).ravel())
panel_values = np.concatenate(panel_values)
global_map_vmin = float(np.nanmin(panel_values))
global_map_vmax = float(np.nanmax(panel_values))

try:
    plot_station_with_map(
        map_df,
        pm25_panels,
        vmin=global_map_vmin,
        vmax=global_map_vmax,
        ncols=macro_r,
        figsize=(5.6 * macro_r, 4.8 * (config['n_delays'] + 1)),
    )
    plot_station_with_map(
        map_df,
        o3_panels,
        vmin=global_map_vmin,
        vmax=global_map_vmax,
        ncols=macro_r,
        figsize=(5.6 * macro_r, 4.8 * (config['n_delays'] + 1)),
    )
except ModuleNotFoundError as exc:
    print('Skipping station map plots because an optional dependency is missing:', exc)



## Block 12. 构造粗粒化函数与宏观变量

利用前 $r$ 个左奇异方向，定义当前侧宏观变量：

$$
z_t = U_r^{\top} C_{00}^{-1/2} \tilde X_t.
$$

在行向量约定下，这表示先对白化后的延迟观测做投影，再得到前 $r$ 个宏观变量。本部分调用 `build_macro_from_kbar(...)`，其中 `r` 默认取 `selected_r`，也可切换到手动 `rank`。


In [ ]:

macro = build_macro_from_kbar(
    U=U_bar,
    S=S_bar,
    Vt=Vt_bar,
    C00_inv_sqrt=C00_inv_sqrt,
    X=X_pairs,
    r=macro_r,
    feature_names=raw_names,
    Y=Y_pairs,
    C11_inv_sqrt=C11_inv_sqrt,
    center=False,
)

artifacts['macro'] = macro
artifacts['metrics']['macro_r'] = macro_r
artifacts['metrics']['macro_r_source'] = macro_r_source

print('Macro current shape:', macro['z_current'].shape)
print('Macro future shape:', None if macro['z_future'] is None else macro['z_future'].shape)
print('Top features per macro component:')
for idx, items in enumerate(macro['dominant_features'], start=1):
    print(f'  Component {idx}:')
    display(pd.DataFrame(items))



## Block 13. 宏观数据与宏微观对比

这一部分展示：

- 宏观变量时间序列；
- 若干微观变量与宏观变量的标准化对照图。

这里展示的宏观变量维数由 `macro_r` 决定，并会在图前打印所采用的来源。


In [ ]:

z_current = macro['z_current']
macro_names = [f'z_{i+1}' for i in range(macro_r)]

plt.figure(figsize=(12, 4))
for i in range(macro_r):
    plt.plot(z_current[:, i], linewidth=1.3, label=macro_names[i])
plt.title('Macro time series')
plt.xlabel('Pair index')
plt.ylabel('Macro value')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
for idx in selected_micro_indices:
    plt.plot(standardize_for_plot(X_pairs[:, idx]), linewidth=1.0, alpha=0.8, label=f'micro: {raw_names[idx]}')
for i in range(macro_r):
    plt.plot(standardize_for_plot(z_current[:, i]), linewidth=1.8, linestyle='--', label=f'macro: {macro_names[i]}')
plt.title('Micro vs macro (standardized for visualization only)')
plt.xlabel('Pair index')
plt.ylabel('Standardized value')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()



## Block 14. 统一摘要输出

最后，把空气实验的关键结果整理成两组摘要：

- 结构与矩阵信息；
- 理论定义 12/13 下的可逆性与 CE 指标，以及保留的补充指标。


In [ ]:

summary_dict = {
    'experiment': {
        'name': config['experiment_name'],
        'observable_mode': config['observable_mode'],
        'variables': config['variables'],
        'subset_mode': config['subset']['subset_mode'],
        'alpha': config['alpha'],
        'eps': config['eps'],
        'ridge': config['ridge'],
    },
    'data': {
        'station_count': air_matrix['n_stations'],
        'raw_shape': x_data_raw.shape,
        'prepared_shape': x_data_fit.shape,
        'lifted_shape': x_data_lift.shape,
        'lift_function': config['observable_mode'],
        'pair_count': X_pairs.shape[0],
        'dt_fit': dt_fit,
        'tau': tau,
        'lag_steps': config['lag_steps'],
        'n_delays': config['n_delays'],
        'delay_interval': config['delay_interval'],
    },
    'matrices': {
        'C00_shape': C00.shape,
        'C01_shape': C01.shape,
        'C11_shape': C11.shape,
        'K_raw_shape': K_raw.shape,
        'K_bar_shape': K_bar.shape,
        'D_K_shape': D_K.shape,
        'N_K_shape': N_K.shape,
        'sigma_max_K_bar': metrics['sigma_max'],
        'effective_rank': effective_rank_gamma,
    },
    'scores': {
        'G_alpha_K': G_alpha_K,
        'EC': EC,
        'Gamma_alpha_K': Gamma_alpha_K,
        'gamma_alpha_K': gamma_alpha_K,
        'manual_r': manual_r,
        'delta_gamma_manual_r': delta_gamma_manual_r,
        'selected_r': selected_r,
        'delta_gamma_selected_r': delta_gamma_selected_r,
        'macro_r': macro_r,
        'macro_r_source': macro_r_source,
        'top_r_singular_values': S_bar[:macro_r],
        'delta_gamma_by_r': delta_gamma_by_r,
    },
}

print_summary(summary_dict)
